In [2]:
# ==========================================
# CELL 1: Khởi tạo môi trường, kết nối DuckDB và trích xuất Metadata nguồn
# ==========================================

!pip -q install duckdb pandas openpyxl graphviz

from google.colab import drive
from pathlib import Path
from IPython.display import display
import duckdb
import pandas as pd
import zipfile

# Mount Google Drive
drive.mount("/content/drive")

# Thiết lập đường dẫn thư mục dự án và thư mục xuất báo cáo
ROOT = Path("/content/drive/MyDrive/Data_Lakehouse_Project")
OUT = ROOT / "exports" / "schema_mapping_and_lineage"
OUT.mkdir(parents=True, exist_ok=True)

# Tìm kiếm các file cơ sở dữ liệu DuckDB trong dự án
candidates = list(ROOT.rglob("*.duckdb"))

print("Danh sách cơ sở dữ liệu khả dụng:")
for p in candidates:
    print(p)

DB_PATH = next(
    (p for p in candidates if p.name == "credit_lakehouse.duckdb"),
    None
)

if DB_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy tập tin credit_lakehouse.duckdb. "
        "Vui lòng kiểm tra lại đường dẫn cơ sở dữ liệu trong Data Lakehouse."
    )

# Kết nối DuckDB ở chế độ chỉ đọc (Read-only)
con = duckdb.connect(str(DB_PATH), read_only=True)

# Trích xuất danh mục bảng/view (Data Catalog)
catalog = con.execute("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN (
    'information_schema',
    'pg_catalog'
)
ORDER BY table_schema, table_name
""").df()

# Trích xuất chi tiết danh mục cột và kiểu dữ liệu
columns = con.execute("""
SELECT
    table_schema,
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema NOT IN (
    'information_schema',
    'pg_catalog'
)
ORDER BY table_schema, table_name, ordinal_position
""").df()

display(catalog)

print("Cơ sở dữ liệu đang kết nối:", DB_PATH)
print("Thư mục xuất dữ liệu:", OUT)

Mounted at /content/drive
Danh sách cơ sở dữ liệu khả dụng:
/content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb


,table_schema,table_name,table_type
0,bronze,application_test,VIEW
1,bronze,application_train,VIEW
2,bronze,bureau,VIEW
3,bronze,bureau_balance,VIEW
4,bronze,credit_card_balance,VIEW
5,bronze,installments_payments,VIEW
6,bronze,pos_cash_balance,VIEW
7,bronze,previous_application,VIEW
8,gold,fct_application_train,BASE TABLE
9,gold,feature_mart_application_train,BASE TABLE


Cơ sở dữ liệu đang kết nối: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb
Thư mục xuất dữ liệu: /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage


In [3]:

# ==========================================
# CELL 2: Xây dựng bảng ánh xạ Source-to-Target Mapping (STTM) và kiểm tra Metadata
# ==========================================

import re

mapping_rows = []

def add_mapping(group, source, source_col, target, target_col,
                dtype, transform, missing, test):
    mapping_rows.append({
        "group": group,
        "source_table": source,
        "source_column": source_col,
        "target_model": target,
        "target_column": target_col,
        "target_type": dtype,
        "join_filter_aggregation": transform,
        "null_special_handling": missing,
        "proposed_test": test
    })


# ==========================================
# 1. Ánh xạ các trường khóa (Primary Key / Foreign Key)
# ==========================================

for source, key, target in [
    ("application_train", "SK_ID_CURR", "fact_application"),
    ("application_test", "SK_ID_CURR", "fact_application"),
    ("previous_application", "SK_ID_PREV", "fact_previous_application"),
    ("previous_application", "SK_ID_CURR", "fact_previous_application"),
    ("bureau", "SK_ID_BUREAU", "fact_bureau_credit"),
    ("bureau", "SK_ID_CURR", "fact_bureau_credit"),
    ("bureau_balance", "SK_ID_BUREAU", "fact_bureau_monthly"),
    ("credit_card_balance", "SK_ID_PREV", "fact_credit_card_monthly"),
    ("pos_cash_balance", "SK_ID_PREV", "fact_pos_cash_monthly"),
    ("installments_payments", "SK_ID_PREV", "fact_installment_period")
]:
    add_mapping(
        "KEY", source, key, target, key, "BIGINT",
        "TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ",
        "Giữ unmatched; thiếu khóa grain -> quarantine",
        "not_null; unique/composite_unique; relationship_audit"
    )


# ==========================================
# 2. Ánh xạ các chỉ số tài chính / tiền tệ (Monetary Features)
# ==========================================

for source, field, target in [
    ("application_train", "AMT_CREDIT", "fact_application"),
    ("application_train", "AMT_ANNUITY", "fact_application"),
    ("application_train", "AMT_GOODS_PRICE", "fact_application"),
    ("application_train", "AMT_INCOME_TOTAL", "fact_application"),
    ("previous_application", "AMT_APPLICATION", "fact_previous_application"),
    ("previous_application", "AMT_CREDIT", "fact_previous_application"),
    ("bureau", "AMT_CREDIT_SUM", "fact_bureau_credit"),
    ("bureau", "AMT_CREDIT_SUM_DEBT", "fact_bureau_credit"),
    ("credit_card_balance", "AMT_BALANCE", "fact_credit_card_monthly"),
    ("credit_card_balance", "AMT_CREDIT_LIMIT_ACTUAL", "fact_credit_card_monthly"),
    ("installments_payments", "AMT_PAYMENT", "fact_installment_period"),
    ("installments_payments", "AMT_INSTALMENT", "fact_installment_period")
]:
    add_mapping(
        "MONEY", source, field, target, field,
        "DECIMAL(20,4)", "TRY_CAST; tổng hợp đúng grain",
        "Không tự thay NULL bằng 0; đánh dấu lỗi cast",
        "cast_valid; null_rate; reconciliation"
    )

# Chuẩn hóa tên cột và quy tắc tổng hợp thanh toán theo kỳ
for row in mapping_rows:
    if row["source_table"] != "installments_payments":
        continue

    if row["source_column"] == "AMT_PAYMENT":
        row["target_column"] = "total_paid"
        row["join_filter_aggregation"] = (
            "SUM(TRY_CAST(AMT_PAYMENT AS DECIMAL(20,4))); "
            "GROUP BY SK_ID_PREV, NUM_INSTALMENT_VERSION, "
            "NUM_INSTALMENT_NUMBER"
        )
        row["null_special_handling"] = (
            "Giữ NULL; không tự thay AMT_PAYMENT thiếu bằng 0"
        )
        row["proposed_test"] = (
            "cast_valid; null_rate; payment_reconciliation"
        )

    elif row["source_column"] == "AMT_INSTALMENT":
        row["target_column"] = "installment_due"
        row["join_filter_aggregation"] = (
            "MAX(TRY_CAST(AMT_INSTALMENT AS DECIMAL(20,4))); "
            "chỉ áp dụng khi nghĩa vụ trong cùng kỳ nhất quán"
        )
        row["null_special_handling"] = (
            "Giữ NULL; kiểm tra nhiều giá trị nghĩa vụ trong cùng kỳ"
        )
        row["proposed_test"] = (
            "due_consistency; null_rate; reconciliation"
        )


# ==========================================
# 3. Ánh xạ thuộc tính trạng thái (Status / Categorical Attributes)
# ==========================================

for source, field, target in [
    ("bureau", "CREDIT_ACTIVE", "fact_bureau_credit"),
    ("bureau_balance", "STATUS", "fact_bureau_monthly"),
    ("previous_application", "NAME_CONTRACT_STATUS", "fact_previous_application"),
    ("pos_cash_balance", "NAME_CONTRACT_STATUS", "fact_pos_cash_monthly")
]:
    add_mapping(
        "STATUS", source, field, target, field,
        "VARCHAR", "TRIM; ánh xạ theo source_domain",
        "Giữ mã gốc; không đồng nhất C/X với DPD=0",
        "accepted_values; mapping_coverage"
    )


# ==========================================
# 4. Ánh xạ thời gian tương đối (Relative Time Features)
# ==========================================

for source in [
    "bureau_balance",
    "credit_card_balance",
    "pos_cash_balance"
]:
    add_mapping(
        "TIME", source, "MONTHS_BALANCE",
        "dim_relative_month", "relative_month",
        "INTEGER", "TRY_CAST; kết hợp month_domain",
        "Không suy diễn ngày lịch",
        "cast_valid; composite_grain"
    )

for field in [
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT"
]:
    add_mapping(
        "TIME", "installments_payments", field,
        "int_payment_event_clean", field,
        "INTEGER", "TRY_CAST; giữ ngày tương đối",
        "Giữ NULL",
        "cast_valid; null_rate"
    )


# ==========================================
# 5. Ánh xạ các chỉ số dẫn xuất và Surrogate Keys
# ==========================================

derived = [
    # A. Đặc trưng tài chính và khách hàng
    (
        "application_train",
        "AMT_CREDIT / AMT_GOODS_PRICE",
        "mart_credit_risk_features",
        "credit_goods_ratio",
        "DOUBLE",
        "AMT_CREDIT / NULLIF(AMT_GOODS_PRICE, 0)",
        "Thiếu đầu vào hoặc mẫu số không dương -> NULL",
        "formula_test; denominator_test"
    ),
    (
        "application_train",
        "AMT_ANNUITY / AMT_INCOME_TOTAL",
        "mart_credit_risk_features",
        "annuity_income_ratio",
        "DOUBLE",
        "AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0)",
        "Thiếu đầu vào hoặc mẫu số không dương -> NULL",
        "formula_test; denominator_test"
    ),
    (
        "application_train",
        "DAYS_BIRTH",
        "mart_credit_risk_features",
        "age_years",
        "DOUBLE",
        "-DAYS_BIRTH / 365.25",
        "Giữ NULL khi thiếu DAYS_BIRTH",
        "formula_test; range_test"
    ),
    (
        "credit_card_balance",
        "AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL",
        "mart_credit_risk_features",
        "credit_utilization_ratio",
        "DOUBLE",
        "AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)",
        "Thiếu đầu vào hoặc hạn mức không dương -> NULL",
        "formula_test; denominator_test"
    ),

    # B. Surrogate Keys
    (
        "application_train",
        "SK_ID_CURR",
        "fact_application",
        "application_key",
        "VARCHAR",
        "Tạo Surrogate Key bằng hash hoặc sequence từ SK_ID_CURR",
        "Thiếu khóa nghiệp vụ -> quarantine",
        "not_null; unique; key_consistency"
    ),
    (
        "bureau",
        "SK_ID_BUREAU",
        "fact_bureau_credit",
        "bureau_credit_key",
        "VARCHAR",
        "Tạo Surrogate Key từ SK_ID_BUREAU",
        "Thiếu khóa nghiệp vụ -> quarantine",
        "not_null; unique; key_consistency"
    ),
    (
        "installments_payments",
        "SK_ID_PREV + NUM_INSTALMENT_VERSION + NUM_INSTALMENT_NUMBER",
        "fact_installment_period",
        "installment_period_key",
        "VARCHAR",
        "Tạo Surrogate Key từ tổ hợp khóa kỳ trả góp",
        "Thiếu thành phần khóa -> quarantine",
        "not_null; unique; composite_grain"
    ),

    # C. Biến dẫn xuất thanh toán theo kỳ
    (
        "installments_payments",
        "MAX(DAYS_ENTRY_PAYMENT) - DAYS_INSTALMENT",
        "fact_installment_period",
        "days_late_last_payment",
        "INTEGER",
        "MAX(DAYS_ENTRY_PAYMENT) - DAYS_INSTALMENT "
        "sau khi tổng hợp theo kỳ",
        "Thiếu ngày đến hạn hoặc ngày thanh toán -> NULL",
        "date_consistency; formula_test"
    ),
    (
        "installments_payments",
        "CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END",
        "fact_installment_period",
        "has_late_payment",
        "INTEGER",
        "Gán 1 nếu có ít nhất một giao dịch trả sau hạn "
        "trong kỳ; 0 nếu đủ dữ liệu và không có trả trễ",
        "Không đủ thông tin ngày để xác định -> NULL",
        "accepted_values; date_consistency"
    )
]

for (
    source,
    expression,
    target,
    output,
    dtype,
    rule,
    missing,
    test
) in derived:
    add_mapping(
        "SURROGATE_KEY" if output.endswith("_key") else "DERIVED",
        source,
        expression,
        target,
        output,
        dtype,
        rule,
        missing,
        test
    )


# ==========================================
# 6. Kiểm tra Metadata từ lớp Bronze
# ==========================================

mapping_df = pd.DataFrame(mapping_rows)

source_columns = {
    (r.table_name.lower(), r.column_name.upper())
    for r in columns.itertuples()
    if r.table_schema.lower() == "bronze"
}

SQL_KEYWORDS = {
    "MAX", "MIN", "SUM", "COUNT", "AVG",
    "CASE", "WHEN", "THEN", "ELSE", "END",
    "NULLIF", "COALESCE", "TRY_CAST",
    "CAST", "AS", "AND", "OR", "NOT",
    "NULL", "DISTINCT", "INTEGER",
    "DOUBLE", "DECIMAL", "VARCHAR"
}

def validate_source_columns(row):
    expression = str(row["source_column"])

    tokens = re.findall(
        r"\b[A-Z][A-Z0-9_]*\b",
        expression.upper()
    )

    fields = [
        token for token in tokens
        if token not in SQL_KEYWORDS
    ]

    if not fields:
        return "CHECK"

    valid = all(
        (row["source_table"].lower(), field)
        in source_columns
        for field in fields
    )

    return "OK" if valid else "CHECK"


mapping_df["metadata_status"] = mapping_df.apply(
    validate_source_columns,
    axis=1
)


# ==========================================
# 7. Kiểm tra và xuất Source-to-Target Mapping
# ==========================================

assert len(mapping_df) == 40, (
    f"STTM phải có 40 quy tắc, hiện có {len(mapping_df)}"
)

assert "fact_payment_event" not in set(
    mapping_df["target_model"]
), "STTM vẫn chứa fact_payment_event"

mapping_df.to_csv(
    OUT / "Source_To_Target_Mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

display(mapping_df)

print("=== KẾT QUẢ SOURCE-TO-TARGET MAPPING ===")
print("Tổng quy tắc Mapping:", len(mapping_df))
print("Số trường Metadata OK:",
      (mapping_df["metadata_status"] == "OK").sum())
print("Số trường cần kiểm tra:",
      (mapping_df["metadata_status"] == "CHECK").sum())

print("\nPhân loại Mapping:")
display(
    mapping_df.groupby("group")
    .size()
    .reset_index(name="mapping_count")
)

print("Đã xuất Source_To_Target_Mapping.csv")


,group,source_table,source_column,target_model,target_column,target_type,join_filter_aggregation,null_special_handling,proposed_test,metadata_status
0,KEY,application_train,SK_ID_CURR,fact_application,SK_ID_CURR,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
1,KEY,application_test,SK_ID_CURR,fact_application,SK_ID_CURR,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
2,KEY,previous_application,SK_ID_PREV,fact_previous_application,SK_ID_PREV,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
3,KEY,previous_application,SK_ID_CURR,fact_previous_application,SK_ID_CURR,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
4,KEY,bureau,SK_ID_BUREAU,fact_bureau_credit,SK_ID_BUREAU,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
5,KEY,bureau,SK_ID_CURR,fact_bureau_credit,SK_ID_CURR,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
6,KEY,bureau_balance,SK_ID_BUREAU,fact_bureau_monthly,SK_ID_BUREAU,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
7,KEY,credit_card_balance,SK_ID_PREV,fact_credit_card_monthly,SK_ID_PREV,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
8,KEY,pos_cash_balance,SK_ID_PREV,fact_pos_cash_monthly,SK_ID_PREV,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK
9,KEY,installments_payments,SK_ID_PREV,fact_installment_period,SK_ID_PREV,BIGINT,TRY_CAST; giữ khóa nguồn; kiểm tra quan hệ,Giữ unmatched; thiếu khóa grain -> quarantine,not_null; unique/composite_unique; relationshi...,OK


=== KẾT QUẢ SOURCE-TO-TARGET MAPPING ===
Tổng quy tắc Mapping: 40
Số trường Metadata OK: 40
Số trường cần kiểm tra: 0

Phân loại Mapping:


,group,mapping_count
0,DERIVED,6
1,KEY,10
2,MONEY,12
3,STATUS,4
4,SURROGATE_KEY,3
5,TIME,5


Đã xuất Source_To_Target_Mapping.csv


In [4]:

# ==========================================
# CELL 3: Xây dựng Danh mục Mô hình dbt (Model Catalog)
# và Vẽ Sơ đồ Dòng Dữ liệu (DAG)
# ==========================================

from graphviz import Digraph

SOURCES = [
    "application_train",
    "application_test",
    "previous_application",
    "bureau",
    "bureau_balance",
    "pos_cash_balance",
    "credit_card_balance",
    "installments_payments"
]

models = []

def register_model(name, schema, deps, purpose):
    models.append({
        "model_name": name,
        "target_schema": schema,
        "materialized": "view" if schema == "silver" else "table",
        "dependencies": deps,
        "purpose": purpose
    })


# ==========================================
# 1. Lớp Staging (Silver Layer)
# ==========================================

for s in SOURCES:
    register_model(
        "stg_" + s,
        "silver",
        ["src_" + s],
        "Chuẩn hóa nguồn"
    )


# ==========================================
# 2. Lớp Intermediate (Silver Layer)
# ==========================================

inter = {
    "int_application_all": [
        "stg_application_train",
        "stg_application_test"
    ],

    "int_bureau_monthly_status": [
        "stg_bureau_balance"
    ],

    "int_bureau_customer_agg": [
        "stg_bureau",
        "int_bureau_monthly_status"
    ],

    "int_previous_customer_agg": [
        "stg_previous_application"
    ],

    "int_pos_cash_customer_agg": [
        "stg_pos_cash_balance"
    ],

    "int_credit_card_customer_agg": [
        "stg_credit_card_balance"
    ],

    "int_payment_event_clean": [
        "stg_installments_payments"
    ],

    "int_installment_period": [
        "int_payment_event_clean"
    ],

    "int_installment_customer_agg": [
        "int_installment_period"
    ],

    "int_relationship_audit": [
        "stg_previous_application",
        "stg_bureau",
        "stg_bureau_balance",
        "stg_pos_cash_balance",
        "stg_credit_card_balance",
        "stg_installments_payments"
    ],

    "int_rejected_records": [
        "int_relationship_audit"
    ]
}

for n, deps in inter.items():
    register_model(
        n,
        "silver",
        deps,
        "Kiểm tra và tổng hợp trước join"
    )


# ==========================================
# 3. Các bảng Chiều (Dimensions - Gold Layer)
# ==========================================

dims = {
    "dim_customer_snapshot": [
        "int_application_all"
    ],

    "dim_product": [
        "stg_previous_application"
    ],

    "dim_status": [
        "stg_previous_application",
        "stg_bureau",
        "int_bureau_monthly_status",
        "stg_pos_cash_balance"
    ],

    "dim_source": [],

    "dim_relative_month": [
        "stg_bureau_balance",
        "stg_pos_cash_balance",
        "stg_credit_card_balance"
    ]
}

for n, deps in dims.items():
    register_model(
        n,
        "gold",
        deps,
        "Dimension"
    )


# ==========================================
# 4. Các bảng Sự kiện (Facts - Gold Layer)
# Thiết kế mới: Chỉ giữ 7 Fact
# ==========================================

facts = {
    "fact_application": [
        "int_application_all",
        "dim_customer_snapshot"
    ],

    "fact_previous_application": [
        "stg_previous_application",
        "dim_customer_snapshot",
        "dim_product",
        "dim_status"
    ],

    "fact_bureau_credit": [
        "stg_bureau",
        "dim_customer_snapshot",
        "dim_status"
    ],

    "fact_bureau_monthly": [
        "int_bureau_monthly_status",
        "dim_relative_month",
        "dim_status"
    ],

    "fact_pos_cash_monthly": [
        "stg_pos_cash_balance",
        "dim_customer_snapshot",
        "dim_relative_month",
        "dim_status"
    ],

    "fact_credit_card_monthly": [
        "stg_credit_card_balance",
        "dim_customer_snapshot",
        "dim_relative_month"
    ],

    "fact_installment_period": [
        "int_installment_period",
        "dim_customer_snapshot"
    ]
}

for n, deps in facts.items():
    register_model(
        n,
        "gold",
        deps,
        "Fact theo grain đã chốt"
    )


# ==========================================
# 5. Kiểm tra cấu trúc Fact
# ==========================================

expected_facts = {
    "fact_application",
    "fact_previous_application",
    "fact_bureau_credit",
    "fact_bureau_monthly",
    "fact_pos_cash_monthly",
    "fact_credit_card_monthly",
    "fact_installment_period"
}

assert set(facts.keys()) == expected_facts, (
    "Danh sách Fact không khớp thiết kế Chương 3"
)

assert len(facts) == 7, (
    f"Thiết kế yêu cầu 7 Fact, hiện có {len(facts)}"
)

assert "fact_payment_event" not in facts, (
    "fact_payment_event phải được loại bỏ khỏi Gold"
)

assert "int_payment_event_clean" in inter, (
    "Phải giữ dữ liệu giao dịch chi tiết tại Silver"
)

assert "int_installment_period" in inter, (
    "Thiếu mô hình tổng hợp thanh toán theo kỳ"
)


# ==========================================
# 6. Các bảng Data Mart (Marts - Gold Layer)
# ==========================================

marts = {
    "mart_credit_risk_features": [
        "fact_application",
        "int_bureau_customer_agg",
        "int_previous_customer_agg",
        "int_pos_cash_customer_agg",
        "int_credit_card_customer_agg",
        "int_installment_customer_agg"
    ],

    "mart_credit_risk_kpi": [
        "fact_application",
        "fact_previous_application",
        "fact_bureau_credit"
    ],

    "mart_data_quality": [
        "int_relationship_audit",
        "int_rejected_records"
    ]
}

for n, deps in marts.items():
    register_model(
        n,
        "gold",
        deps,
        "Mart"
    )


# ==========================================
# 7. Xây dựng Model Catalog
# ==========================================

model_df = pd.DataFrame(models)

# Đối chiếu catalog thiết kế với inventory vật lý DuckDB.
# Việc tìm thấy bảng trong DB không tự chứng minh lần dbt build gần nhất đã thành công.
physical_inventory = {
    (r.table_schema.lower(), r.table_name.lower()): r.table_type
    for r in catalog.itertuples()
}

# Ánh xạ tên logic sang tên vật lý khác, kèm giới hạn phạm vi.
physical_aliases = {
    ("silver", "int_application_all"): (
        "silver", "int_application_train_prep",
        "Partial mapping: physical model contains application_train only; application_test is not combined."
    ),
    ("gold", "fact_application"): (
        "gold", "fct_application_train",
        "Partial mapping: physical fact contains 307,511 train records; test remains separate."
    ),
    ("gold", "mart_credit_risk_features"): (
        "gold", "feature_mart_application_train",
        "Partial mapping: physical feature mart is application-train focused; not the full multi-source feature mart."
    ),
}

def resolve_physical_model(row):
    logical_key = (row["target_schema"].lower(), row["model_name"].lower())

    if logical_key in physical_inventory:
        return pd.Series({
            "physical_model": f"{logical_key[0]}.{logical_key[1]}",
            "physical_materialized": physical_inventory[logical_key],
            "implementation_status": "FOUND_EXACT",
            "scope_note": "Tên logic và tên vật lý khớp trong inventory DuckDB.",
            "build_evidence": "Inventory chỉ xác nhận bảng/view đang tồn tại; xem run_results để kết luận build."
        })

    alias = physical_aliases.get(logical_key)
    if alias:
        schema, physical_name, note = alias
        physical_key = (schema.lower(), physical_name.lower())
        if physical_key in physical_inventory:
            return pd.Series({
                "physical_model": f"{physical_key[0]}.{physical_key[1]}",
                "physical_materialized": physical_inventory[physical_key],
                "implementation_status": "FOUND_ALIAS_PARTIAL_SCOPE",
                "scope_note": note,
                "build_evidence": "Inventory chỉ xác nhận bảng/view đang tồn tại; xem run_results để kết luận build."
            })

    return pd.Series({
        "physical_model": "",
        "physical_materialized": "",
        "implementation_status": "PROPOSED_NOT_FOUND",
        "scope_note": "Chưa có tên vật lý tương ứng trong inventory DuckDB.",
        "build_evidence": "Chưa có bằng chứng vật lý trong inventory snapshot này."
    })

physical_resolution = model_df.apply(resolve_physical_model, axis=1)
model_df = pd.concat([model_df, physical_resolution], axis=1)


# ==========================================
# 8. Xây dựng danh sách quan hệ phụ thuộc
# ==========================================

edges = [
    (dep, row["model_name"])
    for row in models
    for dep in row["dependencies"]
]

lineage_df = pd.DataFrame(
    edges,
    columns=["upstream", "downstream"]
)


# ==========================================
# 9. Kiểm tra tính nhất quán của Model và DAG
# ==========================================

assert len(model_df) == 34, (
    f"Catalog dự kiến 34 Model, hiện có {len(model_df)}"
)

assert "fact_payment_event" not in set(
    model_df["model_name"]
), "Model Catalog vẫn chứa fact_payment_event"

assert not lineage_df.isin(
    ["fact_payment_event"]
).any().any(), "DAG vẫn chứa fact_payment_event"

model_names = set(model_df["model_name"])
source_nodes = {"src_" + s for s in SOURCES}

unknown_dependencies = set(
    lineage_df["upstream"]
) - model_names - source_nodes

assert not unknown_dependencies, (
    f"DAG có phụ thuộc chưa định nghĩa: {unknown_dependencies}"
)


# ==========================================
# 10. Khởi tạo đồ thị Data Lineage DAG
# ==========================================

dot = Digraph(
    "DBT_Transformation_Lineage_DAG"
)

dot.attr(
    rankdir="LR",
    dpi="140",
    splines="spline",
    nodesep="0.35",
    ranksep="0.8"
)

dot.attr(
    "node",
    fontname="Arial",
    fontsize="10"
)

dot.attr(
    "edge",
    color="#8795A5",
    arrowsize="0.6"
)


# ==========================================
# 11. Thêm các nút dữ liệu nguồn
# ==========================================

for s in SOURCES:
    dot.node(
        "src_" + s,
        s,
        shape="cylinder",
        style="filled",
        fillcolor="#FFF0D9",
        color="#D0A46A"
    )


# ==========================================
# 12. Thêm các nút Model
# Phân biệt Model đã triển khai và đề xuất
# ==========================================

for row in model_df.to_dict("records"):
    model_name = row["model_name"]
    model_schema = row["target_schema"]
    status = row["implementation_status"]

    if model_schema == "silver":
        fill = "#DCEBFA"
    else:
        fill = "#E1F2E6"

    if status == "FOUND_EXACT":
        node_style = "rounded,filled"
        node_color = "#4274A8"
        node_label = model_name
    elif status == "FOUND_ALIAS_PARTIAL_SCOPE":
        node_style = "rounded,filled,dashed"
        node_color = "#C27A18"
        node_label = f"{model_name}\n→ {row['physical_model']}"
    else:
        node_style = "rounded,filled,dashed"
        node_color = "#888888"
        node_label = model_name

    dot.node(
        model_name,
        node_label,
        shape="box",
        style=node_style,
        fillcolor=fill,
        color=node_color,
        penwidth="1.5"
    )


# ==========================================
# 13. Thêm các quan hệ phụ thuộc
# ==========================================

for upstream, downstream in edges:
    dot.edge(upstream, downstream)


# ==========================================
# 14. Xuất sơ đồ DAG ra PNG và PDF
# ==========================================

for fmt in ["png", "pdf"]:
    dot.render(
        str(OUT / "DBT_Data_Lineage_DAG"),
        format=fmt,
        cleanup=True
    )


# ==========================================
# 15. Hiển thị kết quả
# ==========================================

display(model_df)

print("=== KẾT QUẢ MODEL CATALOG & DAG ===")
print("Số nguồn dữ liệu:", len(SOURCES))
print("Số Staging:", len(SOURCES))
print("Số Intermediate:", len(inter))
print("Số Dimension:", len(dims))
print("Số Fact:", len(facts))
print("Số Data Mart:", len(marts))
print("Tổng số Model:", len(model_df))
print("Số quan hệ phụ thuộc:", len(lineage_df))

print("\nTrạng thái đối chiếu với inventory DuckDB:")
display(
    model_df.groupby("implementation_status")
    .size()
    .reset_index(name="model_count")
)
exact_count = int((model_df["implementation_status"] == "FOUND_EXACT").sum())
alias_count = int((model_df["implementation_status"] == "FOUND_ALIAS_PARTIAL_SCOPE").sum())
print(
    f"Có {exact_count} model khớp tên vật lý và {alias_count} ánh xạ alias có giới hạn phạm vi "
    f"({exact_count + alias_count} model logic được ánh xạ tới inventory hiện tại)."
)
print("Lưu ý: inventory không xác nhận dbt build thành công; cần đối chiếu run_results.json của lần chạy.")

print("\nĐã xuất sơ đồ:")
print(OUT / "DBT_Data_Lineage_DAG.png")
print(OUT / "DBT_Data_Lineage_DAG.pdf")


,model_name,target_schema,materialized,dependencies,purpose,physical_model,physical_materialized,implementation_status,scope_note,build_evidence
0,stg_application_train,silver,view,[src_application_train],Chuẩn hóa nguồn,silver.stg_application_train,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
1,stg_application_test,silver,view,[src_application_test],Chuẩn hóa nguồn,silver.stg_application_test,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
2,stg_previous_application,silver,view,[src_previous_application],Chuẩn hóa nguồn,silver.stg_previous_application,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
3,stg_bureau,silver,view,[src_bureau],Chuẩn hóa nguồn,silver.stg_bureau,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
4,stg_bureau_balance,silver,view,[src_bureau_balance],Chuẩn hóa nguồn,silver.stg_bureau_balance,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
5,stg_pos_cash_balance,silver,view,[src_pos_cash_balance],Chuẩn hóa nguồn,silver.stg_pos_cash_balance,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
6,stg_credit_card_balance,silver,view,[src_credit_card_balance],Chuẩn hóa nguồn,silver.stg_credit_card_balance,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
7,stg_installments_payments,silver,view,[src_installments_payments],Chuẩn hóa nguồn,silver.stg_installments_payments,VIEW,FOUND_EXACT,Tên logic và tên vật lý khớp trong inventory D...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
8,int_application_all,silver,view,"[stg_application_train, stg_application_test]",Kiểm tra và tổng hợp trước join,silver.int_application_train_prep,VIEW,FOUND_ALIAS_PARTIAL_SCOPE,Partial mapping: physical model contains appli...,Inventory chỉ xác nhận bảng/view đang tồn tại;...
9,int_bureau_monthly_status,silver,view,[stg_bureau_balance],Kiểm tra và tổng hợp trước join,,,PROPOSED_NOT_FOUND,Chưa có tên vật lý tương ứng trong inventory D...,Chưa có bằng chứng vật lý trong inventory snap...


=== KẾT QUẢ MODEL CATALOG & DAG ===
Số nguồn dữ liệu: 8
Số Staging: 8
Số Intermediate: 11
Số Dimension: 5
Số Fact: 7
Số Data Mart: 3
Tổng số Model: 34
Số quan hệ phụ thuộc: 67

Trạng thái đối chiếu với inventory DuckDB:


,implementation_status,model_count
0,FOUND_ALIAS_PARTIAL_SCOPE,3
1,FOUND_EXACT,8
2,PROPOSED_NOT_FOUND,23


Có 8 model khớp tên vật lý và 3 ánh xạ alias có giới hạn phạm vi (11 model logic được ánh xạ tới inventory hiện tại).
Lưu ý: inventory không xác nhận dbt build thành công; cần đối chiếu run_results.json của lần chạy.

Đã xuất sơ đồ:
/content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/DBT_Data_Lineage_DAG.png
/content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/DBT_Data_Lineage_DAG.pdf


In [5]:

# ==========================================
# CELL 4: Quy tắc xử lý bản ghi không khớp (Unmatched Policy)
# và Tạo mẫu truy vấn dbt SQL
# ==========================================

# ==========================================
# 1. Xây dựng Unmatched Policy
# ==========================================

policy_df = pd.DataFrame([
    (
        "bureau_balance",
        3120184,
        "KEEP_UNMATCHED_BUREAU",
        "Giữ SK_ID_BUREAU, không suy diễn customer_key"
    ),
    (
        "credit_card_balance",
        1082816,
        "KEEP_WITH_CUSTOMER",
        "Giữ SK_ID_CURR và đánh dấu unmatched"
    ),
    (
        "installments_payments",
        1250826,
        "KEEP_WITH_CUSTOMER",
        "Giữ SK_ID_CURR và đánh dấu unmatched"
    ),
    (
        "pos_cash_balance",
        340561,
        "KEEP_WITH_CUSTOMER",
        "Giữ SK_ID_CURR và đánh dấu unmatched"
    )
], columns=[
    "source",
    "unmatched_rows_previous_check",
    "policy",
    "handling"
])


# ==========================================
# 2. Khởi tạo thư mục lưu SQL
# ==========================================

sql_dir = OUT / "sql_samples"
sql_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================
# 3. SQL 1: Tổng hợp thanh toán theo kỳ
# Chống Fan-out và cộng lặp AMT_INSTALMENT
# ==========================================

sql_period = """
-- Tổng hợp theo kỳ để tránh cộng lặp AMT_INSTALMENT

WITH payment AS (
    SELECT
        SK_ID_CURR,
        SK_ID_PREV,
        NUM_INSTALMENT_VERSION,
        NUM_INSTALMENT_NUMBER,

        TRY_CAST(
            AMT_PAYMENT AS DECIMAL(20,4)
        ) AS paid,

        TRY_CAST(
            AMT_INSTALMENT AS DECIMAL(20,4)
        ) AS due,

        TRY_CAST(
            DAYS_INSTALMENT AS INTEGER
        ) AS DAYS_INSTALMENT,

        TRY_CAST(
            DAYS_ENTRY_PAYMENT AS INTEGER
        ) AS DAYS_ENTRY_PAYMENT

    FROM {{ ref('stg_installments_payments') }}
)

SELECT
    SK_ID_PREV,
    NUM_INSTALMENT_VERSION,
    NUM_INSTALMENT_NUMBER,

    MAX(SK_ID_CURR) AS SK_ID_CURR,

    COUNT(*) AS payment_events,

    COUNT(DISTINCT SK_ID_CURR) AS customer_count,

    COUNT(DISTINCT due) AS distinct_due_count,

    COUNT(*) FILTER (
        WHERE paid IS NULL
    ) AS null_payment_count,

    SUM(paid) AS total_paid,

    MAX(due) AS installment_due,

    -- Chênh lệch giữa ngày thanh toán cuối và ngày đến hạn
    CASE
        WHEN COUNT(*) FILTER (
            WHERE DAYS_INSTALMENT IS NULL
               OR DAYS_ENTRY_PAYMENT IS NULL
        ) > 0
        OR COUNT(DISTINCT DAYS_INSTALMENT) <> 1
        THEN NULL

        ELSE
            MAX(DAYS_ENTRY_PAYMENT)
            - MAX(DAYS_INSTALMENT)

    END AS days_late_last_payment,

    -- Cờ nhận diện có ít nhất một lần thanh toán trễ
    CASE
        WHEN COUNT(*) FILTER (
            WHERE DAYS_INSTALMENT IS NULL
               OR DAYS_ENTRY_PAYMENT IS NULL
        ) > 0
        OR COUNT(DISTINCT DAYS_INSTALMENT) <> 1
        THEN NULL

        WHEN MAX(
            CASE
                WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT
                THEN 1
                ELSE 0
            END
        ) = 1
        THEN 1

        ELSE 0

    END AS has_late_payment

FROM payment

GROUP BY
    SK_ID_PREV,
    NUM_INSTALMENT_VERSION,
    NUM_INSTALMENT_NUMBER
"""


# ==========================================
# 4. SQL 2: Tổng hợp Bureau theo khách hàng
# ==========================================

sql_bureau = """
-- Tổng hợp Bureau Balance trước khi nối Bureau

WITH monthly AS (
    SELECT
        SK_ID_BUREAU,

        COUNT(*) AS observed_months,

        SUM(
            CASE
                WHEN STATUS IN ('1', '2', '3', '4', '5')
                THEN 1
                ELSE 0
            END
        ) AS overdue_months

    FROM {{ ref('stg_bureau_balance') }}

    GROUP BY SK_ID_BUREAU
)

SELECT
    b.SK_ID_CURR,

    COUNT(*) AS bureau_credit_count,

    SUM(
        COALESCE(m.overdue_months, 0)
    ) AS overdue_months

FROM {{ ref('stg_bureau') }} b

LEFT JOIN monthly m
    ON b.SK_ID_BUREAU = m.SK_ID_BUREAU

GROUP BY b.SK_ID_CURR
"""


# ==========================================
# 5. SQL 3: Xử lý Credit Card không khớp Previous
# ==========================================

sql_unmatched = """
-- Giữ bản ghi Credit Card không khớp Previous

SELECT
    c.*,

    CASE
        WHEN p.SK_ID_PREV IS NULL
        THEN 'UNMATCHED_PREVIOUS'
        ELSE 'MATCHED_PREVIOUS'
    END AS relationship_status

FROM {{ ref('stg_credit_card_balance') }} c

LEFT JOIN {{ ref('stg_previous_application') }} p
    ON c.SK_ID_PREV = p.SK_ID_PREV
"""


# ==========================================
# 6. Tạo danh sách SQL mẫu
# ==========================================

samples = {
    "int_installment_period.sql": sql_period,
    "int_bureau_customer_agg.sql": sql_bureau,
    "unmatched_previous.sql": sql_unmatched
}


# ==========================================
# 7. Kiểm tra SQL và xuất file
# ==========================================

assert "SUM(paid) AS total_paid" in sql_period, (
    "SQL thiếu phép tổng hợp AMT_PAYMENT"
)

assert "MAX(due) AS installment_due" in sql_period, (
    "SQL thiếu quy tắc chống cộng lặp AMT_INSTALMENT"
)

assert "days_late_last_payment" in sql_period, (
    "SQL thiếu biến days_late_last_payment"
)

assert "has_late_payment" in sql_period, (
    "SQL thiếu biến has_late_payment"
)

for filename, content in samples.items():
    file_path = sql_dir / filename

    file_path.write_text(
        content.strip() + "\n",
        encoding="utf-8"
    )


# ==========================================
# 8. Hiển thị kết quả
# ==========================================

display(policy_df)

print("=== KẾT QUẢ SQL & UNMATCHED POLICY ===")
print("Số nguồn có bản ghi không khớp:", len(policy_df))
print("Số file SQL mẫu:", len(samples))

print("\nDanh sách file SQL đã xuất:")
for filename in samples:
    print("-", sql_dir / filename)

print("\nQuy tắc thanh toán:")
print("- Grain: SK_ID_PREV + VERSION + NUMBER")
print("- total_paid: SUM(AMT_PAYMENT)")
print("- installment_due: MAX(AMT_INSTALMENT)")
print("- days_late_last_payment: Đã bổ sung")
print("- has_late_payment: Đã bổ sung")


,source,unmatched_rows_previous_check,policy,handling
0,bureau_balance,3120184,KEEP_UNMATCHED_BUREAU,"Giữ SK_ID_BUREAU, không suy diễn customer_key"
1,credit_card_balance,1082816,KEEP_WITH_CUSTOMER,Giữ SK_ID_CURR và đánh dấu unmatched
2,installments_payments,1250826,KEEP_WITH_CUSTOMER,Giữ SK_ID_CURR và đánh dấu unmatched
3,pos_cash_balance,340561,KEEP_WITH_CUSTOMER,Giữ SK_ID_CURR và đánh dấu unmatched


=== KẾT QUẢ SQL & UNMATCHED POLICY ===
Số nguồn có bản ghi không khớp: 4
Số file SQL mẫu: 3

Danh sách file SQL đã xuất:
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/sql_samples/int_installment_period.sql
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/sql_samples/int_bureau_customer_agg.sql
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/sql_samples/unmatched_previous.sql

Quy tắc thanh toán:
- Grain: SK_ID_PREV + VERSION + NUMBER
- total_paid: SUM(AMT_PAYMENT)
- installment_due: MAX(AMT_INSTALMENT)
- days_late_last_payment: Đã bổ sung
- has_late_payment: Đã bổ sung


In [6]:

# ==========================================
# CELL 5: Tổng hợp Danh mục Kiểm thử (Test Catalog)
# và Xuất toàn bộ Báo cáo / Deliverables
# ==========================================

# ==========================================
# 1. Xây dựng Test Catalog
# ==========================================

test_df = pd.DataFrame([
    (
        "not_null_unique",
        "Khóa nghiệp vụ",
        "0 vi phạm"
    ),
    (
        "composite_grain",
        "Fact tháng/kỳ",
        "0 khóa trùng"
    ),
    (
        "payment_reconciliation",
        "Thanh toán",
        "Chênh lệch 0"
    ),
    (
        "due_consistency",
        "Nghĩa vụ theo kỳ",
        "Không mâu thuẫn"
    ),
    (
        "relationship_audit",
        "Các bảng lịch sử",
        "Bảo toàn unmatched"
    ),
    (
        "no_fanout",
        "Feature Mart",
        "1 dòng / SK_ID_CURR"
    ),
    (
        "status_domain",
        "Bureau Balance",
        "Giữ C/X riêng"
    ),
    (
        "surrogate_key_unique",
        "Surrogate Keys",
        "Không NULL và không trùng"
    ),
    (
        "installment_period_grain",
        "fact_installment_period",
        "Một dòng trên mỗi kỳ trả góp"
    ),
    (
        "late_payment_domain",
        "has_late_payment",
        "Giá trị thuộc tập {0,1,NULL}"
    ),
    (
        "payment_date_consistency",
        "days_late_last_payment",
        "Chỉ tính khi dữ liệu ngày hợp lệ"
    )
], columns=[
    "test_name",
    "scope",
    "expected"
])


# ==========================================
# 2. Kiểm tra cấu trúc Model và Mapping
# ==========================================

expected_facts = {
    "fact_application",
    "fact_previous_application",
    "fact_bureau_credit",
    "fact_bureau_monthly",
    "fact_pos_cash_monthly",
    "fact_credit_card_monthly",
    "fact_installment_period"
}

actual_facts = set(facts.keys())

assert actual_facts == expected_facts, (
    "Danh sách Fact không khớp thiết kế Chương 3"
)

assert len(facts) == 7, (
    "Số lượng Fact phải bằng 7"
)

assert "fact_payment_event" not in set(
    model_df["model_name"]
), "Model Catalog vẫn chứa fact_payment_event"

assert "fact_payment_event" not in set(
    mapping_df["target_model"]
), "STTM vẫn chứa fact_payment_event"

assert not lineage_df.isin(
    ["fact_payment_event"]
).any().any(), "DAG vẫn chứa fact_payment_event"


# ==========================================
# 3. Kiểm tra các trường bắt buộc trong STTM
# ==========================================

required_columns = {
    "application_key",
    "bureau_credit_key",
    "installment_period_key",
    "total_paid",
    "installment_due",
    "days_late_last_payment",
    "has_late_payment"
}

mapped_columns = set(
    mapping_df["target_column"]
)

assert required_columns.issubset(mapped_columns), (
    "STTM chưa đủ các cột theo thiết kế mới"
)


# ==========================================
# 4. Kiểm tra số lượng Model
# ==========================================

assert len(model_df) == 34, (
    f"Thiết kế dự kiến 34 Model, hiện có {len(model_df)}"
)

assert len(mapping_df) == 40, (
    f"Thiết kế dự kiến 40 Mapping, hiện có {len(mapping_df)}"
)


# ==========================================
# 5. Thiết lập đường dẫn Excel
# ==========================================

excel_path = OUT / "Data_Lakehouse_Schema_Mapping_Catalog.xlsx"


# ==========================================
# 6. Xuất Excel đa Sheet
# ==========================================

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    export_sheets = {
        "Source Target Mapping": mapping_df,
        "DBT Model Catalog": model_df,
        "DBT Dependencies": lineage_df,
        "Unmatched Policy": policy_df,
        "Test Catalog": test_df
    }

    for sheet, df in export_sheets.items():
        df.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

    from openpyxl.styles import (
        Font,
        PatternFill,
        Alignment,
        Border,
        Side
    )

    from openpyxl.utils import get_column_letter

    # Định dạng Workbook
    for ws in writer.book.worksheets:

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        # Header Style
        for cell in ws[1]:
            cell.fill = PatternFill(
                "solid",
                fgColor="163E68"
            )

            cell.font = Font(
                color="FFFFFF",
                bold=True
            )

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        ws.row_dimensions[1].height = 30

        # Column Width
        for i in range(1, ws.max_column + 1):
            ws.column_dimensions[
                get_column_letter(i)
            ].width = 30

        # Data Cell Style
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True
                )


# ==========================================
# 7. Xuất các file CSV
# ==========================================

mapping_df.to_csv(
    OUT / "Source_To_Target_Mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

model_df.to_csv(
    OUT / "Model_Catalog.csv",
    index=False,
    encoding="utf-8-sig"
)

lineage_df.to_csv(
    OUT / "Model_Dependencies.csv",
    index=False,
    encoding="utf-8-sig"
)

policy_df.to_csv(
    OUT / "Unmatched_Records_Policy.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    OUT / "Test_Catalog.csv",
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 8. Tổng hợp kết quả
# ==========================================

summary_df = pd.DataFrame([
    ("Nguồn dữ liệu", len(SOURCES)),
    ("Staging Models", len(SOURCES)),
    ("Intermediate Models", len(inter)),
    ("Dimension Models", len(dims)),
    ("Fact Models", len(facts)),
    ("Data Marts", len(marts)),
    ("Tổng Models", len(model_df)),
    ("Source-to-Target Mapping", len(mapping_df)),
    ("DAG Dependencies", len(lineage_df)),
    ("Test Catalog", len(test_df)),
    ("SQL Samples", len(samples))
], columns=[
    "Hạng mục",
    "Số lượng"
])

display(summary_df)


# ==========================================
# 9. Kiểm tra file đầu ra
# ==========================================

output_files = [
    excel_path,
    OUT / "Source_To_Target_Mapping.csv",
    OUT / "Model_Catalog.csv",
    OUT / "Model_Dependencies.csv",
    OUT / "Unmatched_Records_Policy.csv",
    OUT / "Test_Catalog.csv",
    OUT / "DBT_Data_Lineage_DAG.png",
    OUT / "DBT_Data_Lineage_DAG.pdf"
]

for filename in samples:
    output_files.append(sql_dir / filename)

missing_files = [
    str(path)
    for path in output_files
    if not path.exists()
]

assert not missing_files, (
    f"Thiếu file đầu ra: {missing_files}"
)


# ==========================================
# 10. Thông báo kết quả
# ==========================================

print("\n" + "=" * 60)
print("HOÀN THÀNH XUẤT SẢN PHẨM MÔ HÌNH HÓA DỮ LIỆU")
print("=" * 60)

print("\nKẾT QUẢ KIỂM TRA:")
print("- 7 Fact Gold: ĐÚNG")
print("- fact_payment_event: ĐÃ LOẠI BỎ")
print("- int_payment_event_clean: ĐÃ GIỮ")
print("- Surrogate Key Mapping: ĐÃ BỔ SUNG")
print("- Payment Derived Features: ĐÃ BỔ SUNG")
print("- SQL chống Fan-out: ĐÃ TẠO")
print("- Model Catalog & DAG: ĐÃ XUẤT")

print("\nDANH SÁCH FILE:")
for path in output_files:
    print("-", path)

print("\nTHƯ MỤC KẾT QUẢ:")
print(OUT)


,Hạng mục,Số lượng
0,Nguồn dữ liệu,8
1,Staging Models,8
2,Intermediate Models,11
3,Dimension Models,5
4,Fact Models,7
5,Data Marts,3
6,Tổng Models,34
7,Source-to-Target Mapping,40
8,DAG Dependencies,67
9,Test Catalog,11



HOÀN THÀNH XUẤT SẢN PHẨM MÔ HÌNH HÓA DỮ LIỆU

KẾT QUẢ KIỂM TRA:
- 7 Fact Gold: ĐÚNG
- fact_payment_event: ĐÃ LOẠI BỎ
- int_payment_event_clean: ĐÃ GIỮ
- Surrogate Key Mapping: ĐÃ BỔ SUNG
- Payment Derived Features: ĐÃ BỔ SUNG
- SQL chống Fan-out: ĐÃ TẠO
- Model Catalog & DAG: ĐÃ XUẤT

DANH SÁCH FILE:
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Data_Lakehouse_Schema_Mapping_Catalog.xlsx
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Source_To_Target_Mapping.csv
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Model_Catalog.csv
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Model_Dependencies.csv
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Unmatched_Records_Policy.csv
- /content/drive/MyDrive/Data_Lakehouse_Project/exports/schema_mapping_and_lineage/Test_Catalog.csv
- /content/drive/MyDrive/Data_Lakehouse_